In [21]:
import numpy as np
from pathlib import Path
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()

from copy import deepcopy

# Progress bar
from tqdm.auto import tqdm
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torchvision import datasets, transforms

import tensorboard as tb
from torch.utils.tensorboard import SummaryWriter

In [22]:
class RNN(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch_size, seq_len, input_size)

        output, hidden = self.rnn(x)

        # hidden: (num_layers, batch_size, hidden_size)
        last_hidden = hidden[-1]   # pak laatste laag

        logits = self.fc(last_hidden)
        return logits

In [23]:
TRAIN_DATA = 'train_data.csv'
TRAIN_ANSWERS = 'train_answers.csv'
TEST_DATA = 'test_data.csv'
df = pd.read_csv(TRAIN_DATA)
print(df.head())

           id                                         FalseSent  \
0  sentence_1                                I sting a mosquito   
1  sentence_2                            A giraffe is a person.   
2  sentence_3  A normal closet is larger than a walk-in closet.   
3  sentence_4                       I like to ride my chocolate   
4  sentence_5                    A GIRL WON THE RACE WITH HORSE   

                                             OptionA  \
0                                A human is a mammal   
1              Giraffes can drink water from a lake.   
2                Walk-in closets are normal closets.   
3           Chocolate is delicious and bikes are not   
4  GIRL HAVE BEAUTIFUL HAIR BUT THE HORSE DOESN'T...   

                                            OptionB  \
0                             A human is omnivorous   
1                   A giraffe is not a human being.   
2           A person can sleep in a walk-in closet.   
3    Chocolate is a food, not a transpor

In [24]:
from collections import Counter

def build_vocab(texts, min_freq=1):
    counter = Counter()
    for text in texts:
        if isinstance(text, str):  # sla NaN over
            counter.update(text.lower().split())

    vocab = {"<pad>": 0, "<unk>": 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def text_to_indices(text, vocab, max_length):
    if not isinstance(text, str):
        return [vocab["<pad>"]] * max_length  # lege tekst → alleen padding

    tokens = text.lower().split()
    indices = [vocab.get(word, vocab["<unk>"]) for word in tokens]

    if len(indices) < max_length:
        indices += [vocab["<pad>"]] * (max_length - len(indices))
    else:
        indices = indices[:max_length]

    return indices

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): # GPU operation have separate seed
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# Additionally, some operations on a GPU are implemented stochastic for efficiency
# We want to ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [25]:
class CommonSenseDataset(data.Dataset):
    def __init__(self, train_data, answer_data, vocab, max_length=20):
        data = pd.read_csv(train_data)
        answers = pd.read_csv(answer_data)
        self.df = data.merge(answers, on='id')
        self.vocab = vocab
        self.max_length = max_length
        self.label_map = {"A": 0, "B": 1, "C": 2}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        # Zet alle 4 teksten om naar indices
        false_sent = text_to_indices(row["FalseSent"], self.vocab, self.max_length)
        opt_a = text_to_indices(row["OptionA"], self.vocab, self.max_length)
        opt_b = text_to_indices(row["OptionB"], self.vocab, self.max_length)
        opt_c = text_to_indices(row["OptionC"], self.vocab, self.max_length)

        # Stack opties: (3, max_length)
        options = torch.tensor([opt_a, opt_b, opt_c], dtype=torch.long)
        false_sent = torch.tensor(false_sent, dtype=torch.long)
        label = self.label_map[row["answer"]]

        return false_sent, options, label



In [26]:
class RNNClassifier(nn.Module):  # Let op: Module met hoofdletter!
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)  # score per optie

    def forward(self, false_sent, options):
        # false_sent: (batch, seq_len)
        # options: (batch, 3, seq_len)
        batch_size = options.size(0)

        scores = []
        fs_emb = self.embedding(false_sent)          # (batch, seq_len, embed_dim)
        _, fs_hidden = self.rnn(fs_emb)               # encode de false sentence

        for i in range(3):
            opt_emb = self.embedding(options[:, i])    # (batch, seq_len, embed_dim)
            _, opt_hidden = self.rnn(opt_emb)
            # Combineer beide representaties
            combined = fs_hidden[-1] + opt_hidden[-1]  # simpele som
            score = self.fc(combined)                  # (batch, 1)
            scores.append(score)

        logits = torch.cat(scores, dim=1)  # (batch, 3)
        return logits

In [30]:
from sklearn.model_selection import train_test_split

train_label, val_label = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
)

print(len(train_label), len(val_label))

6400 1600


In [ ]:
from torch.utils.data import DataLoader

# Verzamel alle teksten om vocab te bouwen
df = pd.read_csv("train_data.csv")
all_texts = (
    df["FalseSent"].tolist()
    + df["OptionA"].tolist()
    + df["OptionB"].tolist()
    + df["OptionC"].tolist()
)
vocab = build_vocab(all_texts, min_freq=2)

# Maak dataset en loader
train_dataset = CommonSenseDataset("train_data.csv", "train_answers.csv", vocab)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader =

# Test het
false_sent, options, label = train_dataset[0]
print(f"FalseSent shape: {false_sent.shape}")   # (50,)
print(f"Options shape:   {options.shape}")       # (3, 50)
print(f"Label:           {label}")               # 0, 1, of 2

FalseSent shape: torch.Size([20])
Options shape:   torch.Size([3, 20])
Label:           2


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# === Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RNNClassifier(
    vocab_size=len(vocab),
    embed_dim=128,        # grootte van de woordvectoren
    hidden_size=256,      # grootte van het RNN geheugen
    num_layers=2          # aantal gestapelde RNN lagen
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# === Training ===
num_epochs = 20

for epoch in range(num_epochs):
    model.train()                    # zet model in trainingsmodus
    total_loss = 0
    correct = 0
    total = 0

    for false_sent, options, labels in train_loader:
        # Verplaats alles naar GPU/CPU
        false_sent = false_sent.to(device)       # (batch, max_length)
        options = options.to(device)              # (batch, 3, max_length)
        labels = labels.to(device)               # (batch,)

        # Forward pass
        logits = model(false_sent, options)       # (batch, 3)
        loss = loss_fn(logits, labels)

        # Backward pass
        optimizer.zero_grad()   # wis oude gradiënten
        loss.backward()         # bereken nieuwe gradiënten
        optimizer.step()        # pas gewichten aan

        # Bijhouden van statistieken
        total_loss += loss.item()
        predictions = logits.argmax(dim=1)        # pik de hoogste score
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total * 100
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.1f}%")

Epoch 1/20 | Loss: 1.0902 | Accuracy: 39.4%
Epoch 2/20 | Loss: 1.0432 | Accuracy: 45.4%
Epoch 3/20 | Loss: 0.9713 | Accuracy: 54.1%
Epoch 4/20 | Loss: 0.8247 | Accuracy: 64.3%
Epoch 5/20 | Loss: 0.7140 | Accuracy: 70.8%
Epoch 6/20 | Loss: 0.6821 | Accuracy: 72.1%
Epoch 7/20 | Loss: 0.6159 | Accuracy: 74.9%
Epoch 8/20 | Loss: 0.5584 | Accuracy: 77.8%
Epoch 9/20 | Loss: 0.5331 | Accuracy: 79.2%
Epoch 10/20 | Loss: 0.4801 | Accuracy: 81.4%
Epoch 11/20 | Loss: 0.4771 | Accuracy: 81.9%
Epoch 12/20 | Loss: 0.4413 | Accuracy: 83.1%
Epoch 13/20 | Loss: 0.4191 | Accuracy: 84.3%
Epoch 14/20 | Loss: 0.3905 | Accuracy: 85.0%
Epoch 15/20 | Loss: 0.3934 | Accuracy: 85.2%
Epoch 16/20 | Loss: 0.3548 | Accuracy: 86.5%
Epoch 17/20 | Loss: 0.3162 | Accuracy: 88.2%
Epoch 18/20 | Loss: 0.3063 | Accuracy: 88.7%
Epoch 19/20 | Loss: 0.2931 | Accuracy: 89.2%
Epoch 20/20 | Loss: 0.3306 | Accuracy: 88.0%


In [ ]:
class CommonSenseTestset(data.Dataset):
    def __init__(self, train_data, answer_data, vocab, max_length=20):
        self.df = pd.read_csv(train_data)
        self.vocab = vocab
        self.max_length = max_length
        self.label_map = {"A": 0, "B": 1, "C": 2}

        if answer_data is not None:
            answers = pd.read_csv(answer_data)
            self.df = self.df.merge(answers, on='id')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        false_sent = text_to_indices(row["FalseSent"], self.vocab, self.max_length)
        opt_a = text_to_indices(row["OptionA"], self.vocab, self.max_length)
        opt_b = text_to_indices(row["OptionB"], self.vocab, self.max_length)
        opt_c = text_to_indices(row["OptionC"], self.vocab, self.max_length)

        options = torch.tensor([opt_a, opt_b, opt_c], dtype=torch.long)
        false_sent = torch.tensor(false_sent, dtype=torch.long)

        if "answer" in self.df.columns:
            label = self.label_map[row["answer"]]
            return false_sent, options, label
        else:
            return false_sent, options



In [50]:
# Aanroepen moet zo:
test_dataset = CommonSenseTestset(
    train_data=TEST_DATA,
    answer_data=None,
    vocab=vocab,
    max_length=50
)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)  # NIET shufflen bij test

In [51]:
# === Voorspellingen genereren ===
model.eval()                          # zet model in evaluatiemodus
label_map_reverse = {0: "A", 1: "B", 2: "C"}
all_predictions = []

with torch.no_grad():                 # geen gradiënten nodig bij test
    for false_sent, options in test_loader:
        false_sent = false_sent.to(device)
        options = options.to(device)

        logits = model(false_sent, options)
        predictions = logits.argmax(dim=1)   # (batch,)

        # Zet getallen terug naar letters
        for pred in predictions:
            all_predictions.append(label_map_reverse[pred.item()])

# === Opslaan als CSV ===
test_df = pd.read_csv("test_data.csv")
result = pd.DataFrame({
    "id": test_df["id"],
    "answer": all_predictions
})
result.to_csv("sample.csv", index=False)

print(result.head())
# Output:
#           id answer
# 0  sentence_1      B
# 1  sentence_2      A
# 2  sentence_3      C
# ...

           id answer
0  sentence_1      B
1  sentence_2      C
2  sentence_3      A
3  sentence_4      B
4  sentence_5      B
